# RAG 기본 프로세스

### 라이브러리 설치

In [1]:
!pip install -q ollama==0.6.1
!pip install -q langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6
!pip install -q chromadb==1.3.7 pypdf==6.4.2 sentence-transformers==5.2.0
!pip install -q openai==2.12.0


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

# langchain, langchain-community, langchain-openai: RAG 구성의 핵심
# chromadb: 벡터 데이터베이스
# pypdf: PDF 로딩
# sentence-transformers: 한국어 임베딩
# openai: GPT/임베딩 연동
# llama-cpp-python, ollama: 로컬/서버형 모델
# ★ llama-cpp-python 및 ollama는 필요 시

# pip install llama-cpp-python==0.3.16



     ---------------------------------------- 0.0/50.7 MB ? eta -:--:--
     --------------- ---------------------- 21.0/50.7 MB 110.7 MB/s eta 0:00:01
     -------------------------- ------------ 33.8/50.7 MB 85.8 MB/s eta 0:00:01
     --------------------------------------  50.6/50.7 MB 92.0 MB/s eta 0:00:01
     --------------------------------------- 50.7/50.7 MB 68.7 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build llama-cpp-python


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [20 lines of output]
  *** scikit-build-core 0.12.2 using CMake 4.3.2 (wheel)
  *** Configuring CMake...
  2026-04-27 17:06:34,622 - scikit_build_core - WARNING - Can't find a Python library, got libdir=None, ldlibrary=None, multiarch=None, masd=None
  loading initial cache file C:\Users\SMT15\AppData\Local\Temp\tmp_fgphqkn\build\CMakeInit.txt
  -- Building for: NMake Makefiles
  CMake Error at CMakeLists.txt:3 (project):
    Running
  
     'nmake' '-?'
  
    failed with:
  
     no such file or directory
  
  
  CMake Error: CMAKE_C_COMPILER not set, after EnableLanguage
  CMake Error: CMAKE_CXX_COMPILER not set, after EnableLanguage
  -- Configuring incomplete, errors occurred!
  
  *** CMake configuration failed
  [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python

[notice] A new release of pip is available: 25.0.1 

### 라이브러리 선언

In [3]:
# langchain, langchain-community, langchain-openai: RAG 구성핵심
# chromadb: 벡터DB pypdf: PDF sentence-transformers: 한국어 임베딩, ai: GPT연동 cpp/ollama:로컬

import os

# RAG 프로세스 핵심 라이브러리
import langchain, chromadb, pypdf
# PDF 로더
from langchain_community.document_loaders import PyPDFLoader
# 텍스트 청크 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 임베딩모델 OpenAI 라이브러리
from langchain_openai import OpenAIEmbeddings
# 임베딩모델 무료 라이브러리 (허깅페이스)
from langchain_community.embeddings import HuggingFaceEmbeddings
# 벡터 DB 생성 라이브러리
from langchain_community.vectorstores import Chroma
# 모델 GPT 사용 라이브러리
from langchain_openai import ChatOpenAI
# 모델 GGUF 사용 라이브러리
from langchain_community.llms import LlamaCpp
# 모델 OLLAMA 사용 라이브러리
from langchain_community.llms import Ollama
# 임베딩모델 Ollama 라이브러리
from langchain_community.embeddings import OllamaEmbeddings
# 프롬프트 템플릿 라이브러리
from langchain_core.prompts import PromptTemplate
# RAG 체인 구성 라이브러리
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

### 환경 설정 *** 본인 OPENAI_API키 로 설정 필요

In [4]:
# 폴더 구조 설정
DATA_DIR = "dataset"
PDF_PATH = os.path.join(DATA_DIR, "모집요강.pdf")
PERSIST_DIR = "vector_db"
MODEL_DIR = "models"
# OPENAI_APIKEY = "OPENAI_API키"

# LLM 모델 선택: "gpt" 또는 "ollama"
MODEL_TYPE = "ollama"   # "gpt" 또는 "ollama" 중 선택
OLLAMA_MODEL = "gemma4:e2b"  # Ollama 사용 시 모델명

# 폴더 생성
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("PDF 경로:", PDF_PATH)
print("저장 경로:", PERSIST_DIR)

데이터 폴더: dataset
PDF 경로: dataset\모집요강.pdf
저장 경로: vector_db


In [5]:
# 설정값 정의
CHUNK_SIZE = 800        # 청크 크기 (권장: 600-1200)
CHUNK_OVERLAP = 150     # 청크 오버랩 (권장: 100-200)
K_RETRIEVAL = 4         # 검색할 문서 수 (권장: 3-6)

print("설정된 하이퍼파라미터:")
print(f"청크 크기: {CHUNK_SIZE}")
print(f"오버랩: {CHUNK_OVERLAP}")
print(f"검색 수: {K_RETRIEVAL}")

print("\n 설정값 영향:")
print("   - 큰 청크: 문맥 유지↑, 검색 정밀도↓")
print("   - 많은 오버랩: 문맥 단절 방지, 중복↑")
print("   - 높은 K값: 정확도↑, 속도/비용↑")

설정된 하이퍼파라미터:
청크 크기: 800
오버랩: 150
검색 수: 4

 설정값 영향:
   - 큰 청크: 문맥 유지↑, 검색 정밀도↓
   - 많은 오버랩: 문맥 단절 방지, 중복↑
   - 높은 K값: 정확도↑, 속도/비용↑


# 1. 데이터 준비

### 1-1. Data Load

In [7]:
# PDF 로딩 실행
# PDF 로더 생성 및 문서 로딩 (페이지 단위 로딩)
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f" {len(docs)}개 페이지 로딩 완료!")
print(f" 첫 번째 페이지 미리보기 (300자):")
print("-" * 50)
print(docs[0].page_content[:300])
print("-" * 50)
print(f" 메타데이터(page 및 source): {docs[0].metadata}")

 20개 페이지 로딩 완료!
 첫 번째 페이지 미리보기 (300자):
--------------------------------------------------
- 1 -
한국폴리텍대학 서울강서캠퍼스2026학년도 하이테크과정 모집요강   (주소) 서울특별시 강서구 우장산로10길 112 (대표번호) 02-2186-58001 모집학과(직종) 및 모집인원
 ※ 모집학과 및 모집 차수별 모집인원은 대학 사정에 따라 변경될 수 있음 ※ 기회균등 모집인원 미달시 일반선발 인원으로 모집(또한 각 전형 미달 발생 시 우선선발과 일반선발 교차선발) ※ 기회균등 대상자 (「직업교육훈련촉진법」시행령 제10조 해당자) ※ 정원의 110%까지 선발하며, 모집 차수별 미달 인원은 이월하여 다음 차수에 모집(모집1차
--------------------------------------------------
 메타데이터(page 및 source): {'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2018 10.0.0.14515', 'creationdate': '2025-10-27T18:45:54+09:00', 'author': 'Moel', 'moddate': '2025-10-27T18:45:54+09:00', 'pdfversion': '1.4', 'source': 'dataset\\모집요강.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}


### 1-2. Split

In [8]:
# 텍스트 청크 분할 실행
print(" 텍스트 청크 분할 시작...")

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

# 문서 분할 실행
chunks = text_splitter.split_documents(docs)

print(f"청크 분할 결과:")
print(f"원본 페이지 수: {len(docs)}")
print(f"생성된 청크 수: {len(chunks)}")


print(f"\n첫 번째 청크 샘플:")
print("-" * 50)
print(chunks[0].page_content[:200] + "...")
print("-" * 50)
print(f"청크 메타데이터: {chunks[0].metadata}")

 텍스트 청크 분할 시작...
청크 분할 결과:
원본 페이지 수: 20
생성된 청크 수: 46

첫 번째 청크 샘플:
--------------------------------------------------
- 1 -
한국폴리텍대학 서울강서캠퍼스2026학년도 하이테크과정 모집요강   (주소) 서울특별시 강서구 우장산로10길 112 (대표번호) 02-2186-58001 모집학과(직종) 및 모집인원
 ※ 모집학과 및 모집 차수별 모집인원은 대학 사정에 따라 변경될 수 있음 ※ 기회균등 모집인원 미달시 일반선발 인원으로 모집(또한 각 전형 미달 발생 시 우선선발과 ...
--------------------------------------------------
청크 메타데이터: {'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2018 10.0.0.14515', 'creationdate': '2025-10-27T18:45:54+09:00', 'author': 'Moel', 'moddate': '2025-10-27T18:45:54+09:00', 'pdfversion': '1.4', 'source': 'dataset\\모집요강.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}


### 1-3. 임베딩 Store

In [9]:
# 임베딩 모델 선택 및 초기화
# MODEL_TYPE에 따라 임베딩 자동 선택
# ollama: bge-m3 (Ollama 로컸), gpt: OpenAI, 그 외: HuggingFace 한국어
USE_OPENAI_EMBEDDING = True  # GPT 모드일 때 True, 개별 설정 원할 시 직접 변경

if MODEL_TYPE == "ollama":
    # Ollama 로컸 임베딩 (bge-m3)
    print("Ollama bge-m3 임베딩 모델 로딩...")
    embedding_model = OllamaEmbeddings(
        model="bge-m3"
    )
    print("Ollama bge-m3 임베딩 준비 완료")

elif USE_OPENAI_EMBEDDING:
    # OpenAI 임베딩 사용
    print("OpenAI 임베딩 모델 로딩...")
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY
    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
    print("OpenAI 유료 임베딩 준비 완료")

else:
    # 한국어 임베딩 사용
    print("한국어 임베딩 무료 모델 로딩...")
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("무료 임베딩 모델 준비 완료")


Ollama bge-m3 임베딩 모델 로딩...
Ollama bge-m3 임베딩 준비 완료


C:\Users\SMT15\AppData\Local\Temp\ipykernel_23220\906159268.py:9: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding_model = OllamaEmbeddings(


### 벡터 데이터베이스의 역할
- 임베딩 벡터 저장/색인/검색**(KNN)
- 메타데이터와 함께 관리**(source, page)
- RAG에서 검색 단계**의 성능/속도 담당


```
[문서 임베딩] → [컬렉션 저장] → [인덱싱] → [쿼리 임베딩] → [최근접 이웃 검색] → [Top-K 결과]
```

In [10]:
# ChromaDB 생성 및 벡터 저장

print("ChromaDB 벡터 저장소 생성...")
print(f"처리할 청크 수: {len(chunks)}")

# ChromaDB 생성 (임베딩과 함께)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR
)

# 영구 저장 (최신 버전에서는 자동으로 저장됨)
# vectorstore.persist()
print("ChromaDB 영구 저장 완료!")

# 검색기 생성
retriever = vectorstore.as_retriever(
    search_kwargs={"k": K_RETRIEVAL}
)

# 결과 확인
collection_count = vectorstore._collection.count()
print(f"저장된 벡터 수: {collection_count}")
print(f"검색 설정: Top-{K_RETRIEVAL}")

# 테스트 검색 실행 (수정된 부분)
test_query = "면접 일정"

# 방법 1: invoke() 사용 (권장)
test_results = retriever.invoke(test_query)

# 방법 2: vectorstore에서 직접 검색
# test_results = vectorstore.similarity_search(test_query, k=K_RETRIEVAL)

print(f"\n테스트 검색 ('{test_query}'):")
print(f"검색된 문서 수: {len(test_results)}")
if test_results:
    print(f"   첫 번째 결과 (100자): {test_results[0].page_content[:100]}...")


ChromaDB 벡터 저장소 생성...
처리할 청크 수: 46
ChromaDB 영구 저장 완료!
저장된 벡터 수: 46
검색 설정: Top-4

테스트 검색 ('면접 일정'):
검색된 문서 수: 4
   첫 번째 결과 (100자): ■ 면접 안내 - 개인별 면접 시간 및 장소는 면접일 전일 캠퍼스    홈페이지에 공지되며, 개별 공지하지 않음 - 면접 불참자는 불합격 처리됨 ※ 캠퍼스 사정에 따라 면접일을 연...


# 2. 정보 검색 및 텍스트 생성

### 2-1. 모델선언 및 Retrieval

In [11]:
# LLM 모델 선택 및 초기화
# 모델 설정 (gpt/gguf/ollama 중 선택)
# 위 환경설정 셀에서 MODEL_TYPE 조정 가능 ("gpt" 또는 "ollama")
TEMPERATURE = 0.2     # 사실 기반 답변용 (0~1)

print(f"{MODEL_TYPE.upper()} 모델 초기화...")

if MODEL_TYPE == "gpt":
    # GPT 모델 사용
    print("GPT 모델 로딩...")
    # API 키 설정 (실제 사용시 입력 필요)
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY

    llm = ChatOpenAI(
        model="gpt-3.5-turbo",  # 또는 "gpt-4o-mini"
        temperature=TEMPERATURE
    )
    print("GPT 모델 준비 완료")

elif MODEL_TYPE == "gguf":
    # ★ 수정포인트
    # 모델 경로 (실제 사용시 다운로드 필요)
    model_path = "/content/llama-model.gguf"

    llm = LlamaCpp(
        model_path=model_path,
        n_ctx=4096,
        n_threads=2,
        temperature=TEMPERATURE
    )
    print("GGUF 모델 준비 완료")

elif MODEL_TYPE == "ollama":
    # Ollama 모델 사용
    print("Ollama 모델 로딩...")
    llm = Ollama(
        model=OLLAMA_MODEL,
        temperature=TEMPERATURE
    )
    print("Ollama 모델 준비 완료")

print(f"설정된 모델: {MODEL_TYPE.upper()}")
print(f"Temperature: {TEMPERATURE}")

OLLAMA 모델 초기화...
Ollama 모델 로딩...
Ollama 모델 준비 완료
설정된 모델: OLLAMA
Temperature: 0.2


C:\Users\SMT15\AppData\Local\Temp\ipykernel_23220\3178657097.py:36: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [12]:
# 테스트
try:
    test_response = llm.invoke("안녕하세요")

    # ChatModel (GPT)과 LLM의 반환 타입이 다름
    if MODEL_TYPE == "gpt":
        # ChatOpenAI는 객체를 return 하여 content
        response_text = test_response.content
    else:
        # LlamaCpp, Ollama는 문자열 반환
        response_text = test_response

    print(f"모델 테스트 성공: {response_text[:50]}...")

except Exception as e:
    print(f"모델 테스트 실패: {e}")
    print("API 키나 모델 경로를 확인하세요!")


모델 테스트 성공: 안녕하세요! 😊 무엇을 도와드릴까요?...


### 2-2. RAG체인 구성 및 Generation

### RAG 체인: 질문 검색 생성, 프롬프트템플릿, 체인연결

```
[사용자 질문] → [쿼리 임베딩] → [Top-K 검색(Chroma)] → [컨텍스트 구성] → [LLM 응답 생성] → [답변 + 출처]
```

In [13]:
# RAG용 프롬프트 템플릿 정의
PROMPT_TEMPLATE = """
다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
"""

# PromptTemplate 객체 생성
prompt_template = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print("프롬프트 템플릿 생성 완료!")
print("\n템플릿 내용:")
print("-" * 50)
print(PROMPT_TEMPLATE[:200] + "...")
print("-" * 50)

# 템플릿 테스트
test_context = "HKCODE 유튜브 채널은 AI관련 정보공유"
test_question = "HKCODE 유튜브 채널은 누가 운영하나요?"

formatted_prompt = prompt_template.format(
    context=test_context,
    question=test_question
)

print("\n프롬프트 포맷팅 테스트:")
print(formatted_prompt)

프롬프트 템플릿 생성 완료!

템플릿 내용:
--------------------------------------------------

다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
...
--------------------------------------------------

프롬프트 포맷팅 테스트:

다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
HKCODE 유튜브 채널은 AI관련 정보공유

질문: HKCODE 유튜브 채널은 누가 운영하나요?
답변:



In [14]:
# 출처 문서 포함 RAG 체인
print("RAG 체인 구성 중 (출처 문서 반환)...")

# 문서 포맷팅 함수
def format_docs(docs):
    """검색된 문서를 텍스트로 변환"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[출처 {i}] 페이지 {page}\n{content}")
    return "\n\n".join(formatted)

# Step 1: 검색 및 문맥 준비
retrieval_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "source_documents": retriever
    }
)

# Step 2: 답변 생성 체인
answer_chain = (
    {
        "context": lambda x: x["context"],
        "question": lambda x: x["question"]
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

# Step 3: 전체 체인 결합
rag_chain = retrieval_chain | RunnablePassthrough.assign(answer=answer_chain)

print("RAG 체인 생성 완료!")

print("\n체인 구성요소:")
print(f"LLM: {MODEL_TYPE.upper()}")
print(f"Retriever: ChromaDB (K={K_RETRIEVAL})")
print(f"Prompt: 커스텀 템플릿")
print(f"Source Docs: 반환 활성화")


RAG 체인 구성 중 (출처 문서 반환)...
RAG 체인 생성 완료!

체인 구성요소:
LLM: OLLAMA
Retriever: ChromaDB (K=4)
Prompt: 커스텀 템플릿
Source Docs: 반환 활성화


### 질의 테스트

In [15]:
# 질문
question = "스마트금융과 모집 일정은?"

print(f"\n질문: {question}")

# 전체 결과 받기
full_result = rag_chain.invoke(question)

# 각 요소 분리 저장
answer = full_result['answer']              # 답변만
source_docs = full_result['source_documents']  # 출처 문서 리스트
context = full_result['context']            # 포맷된 컨텍스트
question_echo = full_result['question']     # 질문 (확인용)

# 출력
print(f"\n답변:\n{answer}")

print(f"\n참조 문서 ({len(source_docs)}개):")
for i, doc in enumerate(source_docs, 1):
    page = doc.metadata.get('page', '?')
    source = doc.metadata.get('source', '알 수 없음')
    content_preview = doc.page_content[:80].replace('\n', ' ')
    print(f"   [{i}] 페이지 {page}")
    print(f"       {content_preview}...")


질문: 스마트금융과 모집 일정은?

답변:
스마트금융과의 모집 일정은 다음과 같습니다.

*   **모집 1차:** 2025.12.19.(금) 10시
*   **모집 2차:** 2026.02.13.(금) 10시

출처: [출처 3] 페이지 0, [출처 4] 페이지 7-8

참조 문서 (4개):
   [1] 페이지 8
       - 9 - 10 학과 및 기타  <하이테크과정 학과>  <하이테크과정 기타 안내>  ❍ 교육훈련비 전액 국비지원(교육훈련비, 실습비 등)  ❍ ...
   [2] 페이지 7
       발표 모집 1차 2025.12.19.(금) 10시 모집 2차2026.02.13.(금) 10시 유의사항▪ 합격자 발표는 홈페이지를 통해 발표, 본...
   [3] 페이지 0
       ■ 면접 안내 - 개인별 면접 시간 및 장소는 면접일 전일 캠퍼스    홈페이지에 공지되며, 개별 공지하지 않음 - 면접 불참자는 불합격 처리됨...
   [4] 페이지 7
       - 8 - 8 합격자발표 및 충원 9 등록포기 안내 구분 내용등록포기 방법▪ 온라인(홈페이지)으로 본인이 직접 등록포기▪ 등록포기 신청이 승인처...


### 참고. FAST API 연동

In [16]:
!pip install -U nest-asyncio==1.6.0 pyngrok==7.2.4 uvicorn==0.34.2 fastapi==0.115.12
import uvicorn
# 서버 관리용 fastapi 의존 라이브러리
import uvicorn

# fast api 라이브러리
from fastapi import FastAPI

# 데이터프레임 및 수 처리 라이브러리
import pandas as pd
# 인터페이스 데이터 관리를 위한 라이브러리
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.46.0
    Uninstalling uvicorn-0.46.0:
      Successfully uninstalled uvicorn-0.46.0



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
origins = ["*"]

app = FastAPI(title="ML API")

# CORS 미들웨어 추가
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 모든 origin 허용
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

In [18]:
class InDataset(BaseModel):
    question : str

In [20]:
# 테스트 코드
# print(x)
question = "스마트금융과 모집 일정은?"
response = rag_chain.invoke(question)
response1 = response["answer"]
response2 = response["source_documents"]
print(response1)
print(response2)

스마트금융과 모집 일정은 다음과 같습니다.

*   **모집 1차:** 2025.12.19.(금)
*   **모집 2차:** 2026.02.13.(금)

(출처 3)
[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2018 10.0.0.14515', 'pdfversion': '1.4', 'creationdate': '2025-10-27T18:45:54+09:00', 'total_pages': 20, 'page': 8, 'moddate': '2025-10-27T18:45:54+09:00', 'author': 'Moel', 'source': 'dataset\\모집요강.pdf', 'page_label': '9'}, page_content='- 9 -\n10 학과 및 기타  <하이테크과정 학과>\n <하이테크과정 기타 안내>  ❍ 교육훈련비 전액 국비지원(교육훈련비, 실습비 등)  ❍ 교육 기간 중 매주 수업은 월요일~금요일(09:00~17:40) 실시  ❍ 국가기술자격증 취득 기회 부여 \n학과 내용스마트금융☎02-2186-5860▪ 금융과 소프트웨어 기술을 융합한 핀테크 산업 분야의 실무형 개발자 양성▪ (주요교과목) 빅데이터 분석, 블록체인 개발, 금융데이터 분석, 웹 어플리케이션 개발▪ (취득가능자격증) 정보처리기사, 정보처리산업기사, 리눅스마스터, SQLD, OCJP, OCP▪ (취업분야) 데이터분석 기업, 소프트웨어 개발업체, 블록체인 소프트웨어 개발업체 등\n사이버보안☎02-2186-5850▪정보보안 분야의 이론과 실무를 겸비한 정보보안 관제 인력 및 엔지니어 양성▪(주요교과목) 정보보안개론, 응용보안, 시스템보안, 네트워크보안, 정보보안제품운용, 프로그래밍 언어▪(취득가능자격증) 해킹보안전문가 3급, 정보보안산업기사, 개인정보관리사(CPPG), 국제공인정보보안    관리자(CISM), 국제공인정보시스템감사사(CISA), 국제 공인정보시스템보안전문가(CISSP)▪ (취업분야)

In [21]:
@app.post("/predict", status_code=200)
async def predict_tf(x: InDataset):
    print(x)
    question = "스마트금융과 모집 일정은?"
    response = rag_chain.invoke(question)
    response1 = response["answer"]
    response2 = response["source_documents"]
    print(response1)
    print(response2)

    return {"prediction": response1,
            "references": response2 }

@app.get('/')
async def root():
    return {"message": "online"}

In [23]:
# if inColab == True:
#     import nest_asyncio
#     from pyngrok import ngrok
#     import uvicorn

#     auth_token = "371Ga2zptlxD4G8mq2oY7exyIUm_363aLB8Dvt6v6VqAMkzFz"
#     ngrok.set_auth_token(auth_token)
#     ngrokTunnel = ngrok.connect(8000)
#     print("공용 URL", ngrokTunnel.public_url)
#     nest_asyncio.apply()
#     uvicorn.run(app, port=8000)

# else:
import nest_asyncio
nest_asyncio.apply()

# Uvicorn 실행
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9999, log_level="debug")

INFO:     Started server process [23220]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9999 (Press CTRL+C to quit)


INFO:     127.0.0.1:52126 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:52126 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:52126 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:52126 - "GET /openapi.json HTTP/1.1" 200 OK


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\SMT15\.pyenv\pyenv-win\versions\3.12.10\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001EE8862C340> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\SMT15\.pyenv\pyenv-win\versions\3.12.10\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001EE8862C340> is already entered


INFO:     127.0.0.1:50705 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:50705 - "GET /openapi.json HTTP/1.1" 200 OK


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\SMT15\.pyenv\pyenv-win\versions\3.12.10\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001EE8862C340> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\SMT15\.pyenv\pyenv-win\versions\3.12.10\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001EE8862C340> is already entered
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [23220]


### 참고. 대화형 모드

In [ ]:
# 대화형 RAG 체인 생성
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("대화형 RAG 체인 설정...")

# 대화형 프롬프트 템플릿
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질문답변 AI입니다.
다음 컨텍스트와 대화 기록을 참고하여 한국어로 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하세요."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", """컨텍스트:
{context}

질문: {question}""")
])

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[문서 {i} - 페이지 {page}]\n{content}")
    return "\n\n".join(formatted)

# 대화 기록 저장소 (간단한 리스트)
chat_history = []

# 대화형 RAG 체인
def chat_rag(question):
    """대화 기록을 유지하며 답변을 생성하는 함수"""

    # 검색
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # 답변 생성
    response = chat_prompt | llm | StrOutputParser()
    answer = response.invoke({
        "context": context,
        "question": question,
        "chat_history": chat_history
    })

    # 대화 기록에 추가
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return {
        "question": question,
        "answer": answer,
        "source_documents": docs
    }

print("대화형 RAG 체인 준비 완료!")

# 대화 예시 테스트
print("\n대화형 테스트:")
print("="*50)

# 첫 번째 질문
q1 = "지원 자격이 뭐야?"
print(f"질문 1: {q1}")
result1 = chat_rag(q1)
print(f"답변 1: {result1['answer']}")

# 두 번째 연관 질문
q2 = "그럼 제출 서류는 뭐가 필요해?"
print(f"\n질문 2: {q2}")
result2 = chat_rag(q2)
print(f"답변 2: {result2['answer']}")

# 세 번째 질문 (이전 대화 참조)
q3 = "그거 어디에 제출하면 돼?"
print(f"\n질문 3: {q3}")
result3 = chat_rag(q3)
print(f"답변 3: {result3['answer']}")

print("\n대화형 모드 테스트 성공!")

# 대화 기록 확인
print(f"\n 대화 기록 ({len(chat_history)//2}턴):")
for i in range(0, len(chat_history), 2):
    print(f"\n[턴 {i//2 + 1}]")
    print(f": {chat_history[i].content}")
    print(f": {chat_history[i+1].content[:100]}...")


### 참고. 간단 챗봇

In [ ]:
# 직접 질문하기 (출처 포함 버전)
def ask_question_with_source():
    """출처 문서를 포함한 인터랙티브 질문-답변 시스템"""
    print("\n" + "="*60)
    print("PDF RAG 질문-답변 시스템 (출처 포함)")
    print("="*60)
    print("명령어:")
    print("  • 'exit' / 'quit' / '종료' - 시스템 종료")
    print("  • 'detail' / '상세' - 마지막 답변의 상세 정보 보기")
    print("="*60)

    question_count = 0
    last_result = None

    while True:
        question = input("\n질문을 입력하세요: ").strip()

        if question.lower() in ['exit', 'quit', '종료', '나가기']:
            print(f"\n총 {question_count}개의 질문에 답변했습니다.")
            print("시스템을 종료합니다.")
            break

        if question.lower() in ['detail', '상세']:
            if last_result:
                print("\n마지막 답변 상세 정보:")
                print("="*60)
                print(f"질문: {last_result['question']}")
                print(f"\n답변:\n{last_result['answer']}")
                print(f"\n참조 문서 ({len(last_result['source_documents'])}개):")
                for i, doc in enumerate(last_result['source_documents'], 1):
                    page = doc.metadata.get('page', '?')
                    content = doc.page_content[:150].replace('\n', ' ')
                    print(f"\n  [{i}] 페이지 {page}")
                    print(f"      {content}...")
            else:
                print("아직 질문한 내역이 없습니다.")
            continue

        if not question:
            print("질문을 입력해주세요.")
            continue

        try:
            print("검색 및 답변 생성 중...")

            # rag_chain이 출처 포함 버전인 경우
            result = rag_chain.invoke(question)

            question_count += 1
            last_result = {
                'question': question,
                'answer': result['answer'],
                'source_documents': result['source_documents']
            }

            # 결과 출력
            print("\n" + "="*60)
            print("AI 답변:")
            print("="*60)
            print(result['answer'])

            # 참조 문서 간단 표시
            if result['source_documents']:
                sources = []
                for doc in result['source_documents']:
                    page = doc.metadata.get('page', '?')
                    sources.append(f"p.{page}")
                print(f"\n참조: {', '.join(set(sources))}")  # 중복 제거
                print("'detail' 입력 시 상세 정보를 볼 수 있습니다.")

        except Exception as e:
            print(f"오류 발생: {e}")
            print("다시 시도해주세요.")

# 실행
ask_question_with_source()


### 참고 (이후는 패스) GGUF 로컬 모델 설정

In [ ]:
# GGUF 로컬 모델 설정
print("="*60)
print("GGUF 로컬 모델 로딩")
print("="*60)

# Step 1: 패키지 설치 (필요시)
print("필요한 패키지 확인 중...")
try:
    import llama_cpp
    print("llama-cpp-python 이미 설치됨")
except ImportError:
    print("llama-cpp-python 설치 중...")
    !pip install -q llama-cpp-python

# Step 2: 모델 파일 경로 설정
import os

MODEL_PATH = "./pharos_2026_gemma2b.gguf"  # 현재 폴더

# 파일 존재 확인
if os.path.exists(MODEL_PATH):
    file_size = os.path.getsize(MODEL_PATH) / (1024**3)  # GB 단위
    print(f"모델 파일 발견: {MODEL_PATH}")
    print(f"파일 크기: {file_size:.2f} GB")
else:
    print(f"모델 파일을 찾을 수 없습니다: {MODEL_PATH}")
    print("파일 경로를 확인해주세요!")

# Step 3: GGUF 모델 로딩
print("\nGGUF 모델 로딩 중...")
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler

# 콜백 설정 (실시간 출력)
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

# GGUF 모델 초기화
llm_gguf = LlamaCpp(
    model_path=MODEL_PATH,
    n_ctx=4096,           # 컨텍스트 윈도우
    n_threads=2,          # CPU 스레드 수 (환경에 맞게 조정)
    n_gpu_layers=0,       # GPU 사용 안 함 (CPU만)
    temperature=0,        # 결정론적 출력
    max_tokens=512,       # 최대 생성 토큰
    top_p=1,
    callback_manager=callback_manager,
    verbose=False
)

print("GGUF 모델 로딩 완료!")

# Step 4: 간단한 테스트
print("\n모델 테스트:")
try:
    test_response = llm_gguf.invoke("안녕하세요")
    print(f"테스트 성공!")
    print(f"응답: {test_response[:100]}...")
except Exception as e:
    print(f"테스트 실패: {e}")


In [ ]:
# GGUF 기반 RAG 체인 구성
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

print("\n GGUF RAG 체인 구성 중...")

# 프롬프트 템플릿 (한국어 최적화)
prompt_gguf = PromptTemplate.from_template(
    """다음 문맥을 참고하여 질문에 한국어로 답변하세요.
문맥에 없는 내용은 모른다고 답변하세요.

문맥:
{context}

질문: {question}

답변:"""
)

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[문서 {i} - 페이지 {page}]\n{content}")
    return "\n\n".join(formatted)

# RAG 체인 구성
rag_chain_gguf = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_gguf
    | llm_gguf
    | StrOutputParser()
)

print("GGUF RAG 체인 생성 완료!")

print("\n체인 구성요소:")
print(f"  LLM: GGUF (pharos_2026_gemma2b)")
print(f"   Retriever: ChromaDB (K={K_RETRIEVAL})")
print(f"   Prompt: 한국어 최적화")
print(f"   실행: CPU 기반")


In [ ]:
# GGUF RAG 체인 테스트
print("\n" + "="*60)
print("GGUF RAG 체인 테스트")
print("="*60)

# 테스트 질문
test_questions = [
    "지원 자격이 어떻게 되나요?",
    "신청 방법을 알려주세요",
    "제출 서류는 무엇인가요?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n[질문 {i}]")
    print(f"{question}")
    print("-" * 60)

    try:
        # 시간 측정
        import time
        start = time.time()

        answer = rag_chain_gguf.invoke(question)

        elapsed = time.time() - start

        print(f"답변: {answer}")
        print(f"소요 시간: {elapsed:.2f}초")

    except Exception as e:
        print(f"에러: {e}")

    print("="*60)
